####convert the table to a text cause LLM understands the natural language and not table

In [0]:
from pyspark.sql.functions import concat_ws, col, lit, when, to_timestamp, collect_list, window

In [0]:
df_logs = spark.read.format('csv').option('header', True).load('abfss://rag-logs@rahulchurndatalake.dfs.core.windows.net/logs')

In [0]:
df_text = df_logs.withColumn(
    "log_text",
    when(col('status')=='SUCCESS',
    concat_ws(
        " ",
        lit("At"),
        col("timestamp"),
        lit(", the"),
        col("job_name"),
        lit("job completed successfully")
    ))
.otherwise(concat_ws(
        " ",
        lit("At"),
            col("timestamp"),
            lit(", the"),
            col("job_name"),
            lit("job failed due to"),
            col("error_message")
    )))

display(df_text.select("log_text"))

####converting the timestamp from string to timestamp so that we can group the timestamp in 30 mins interval and collect the text as chunks. This is called time based chunking

In [0]:
df_text = df_text.withColumn(
    "timestamp",
    to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss")
)

In [0]:
df_chunked = df_text.groupBy(
    window(col("timestamp"), "30 minutes")
).agg(
    concat_ws(" ", collect_list("log_text")).alias("chunk_text")
)

# df_chunked.display()

In [0]:
df_chunked = df_chunked.select(
    col("window.start").alias("start_time"),
    col("window.end").alias("end_time"),
    col("chunk_text")
)
df_chunked.show()